# AI Model Validation: From Mathematics to Trust
## Episode 05 — Generalization: Why a Model That Performs Well Today Can Fail Tomorrow

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

A model can perform extremely well on the data it has already seen and still fail on new data.

That difference is one of the central scientific problems in Machine Learning.

> **A model is useful only to the extent that its performance generalizes beyond the data used to build it.**

This episode develops generalization from first principles and examines training error, test error, underfitting, overfitting, polynomial complexity, bias–variance trade-off, cross-validation, learning curves, leakage, and uncertainty in model selection.


## 1. Training Performance Is Not Generalization

Suppose a model is fitted using a training dataset.

Its training error measures performance on data that influenced the fitted parameters.

Generalization error concerns expected performance on new observations drawn from the relevant deployment distribution.


A small training error is therefore not sufficient evidence that the model will perform well on unseen data.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys

print("Python :", sys.version.split()[0])
print("System :", platform.system(), platform.release())
print("Machine:", platform.machine())
print("NumPy  :", np.__version__)


## 2. Population Risk and Empirical Risk

Let a loss function be denoted by L.

The **population risk** or expected prediction risk is conceptually:


The expectation is taken over the unknown data-generating distribution.

Because that distribution is not directly available, we estimate performance using finite samples.

The empirical risk on n observed examples is:


Machine Learning therefore operates with an unavoidable distinction:

- what we can measure on available data
- what we want to know about future data


## 3. Generalization Gap

A simple empirical generalization gap can be written as:


A large positive gap can be evidence of overfitting, although interpretation still depends on how the data were sampled and how the model was selected.


## 4. Experiment — A Simple Train/Test Split

We generate data from a known nonlinear process with noise.

The underlying process is not directly given to the fitted model.


In [ ]:
rng = np.random.default_rng(42)

n = 120
X = np.sort(rng.uniform(-3, 3, n))
noise = rng.normal(0, 0.8, n)
y = 0.5 * X**3 - X + noise

split = 80
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("Training samples:", len(X_train))
print("Test samples    :", len(X_test))

plt.figure(figsize=(8,5))
plt.scatter(X_train, y_train, label="Training data")
plt.scatter(X_test, y_test, label="Test data")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Training and Test Data")
plt.legend()
plt.grid(alpha=.25)
plt.show()


The split above is deliberately simple. Later we will examine why one train/test split can itself be unstable.


## 5. Model Complexity

A polynomial model of degree d can be written as:


Increasing d increases the flexibility of the model.

Flexibility can be useful when the true relationship is complex, but excessive flexibility can allow the model to reproduce accidental noise in the training sample.


## 6. Experiment — Underfitting, Appropriate Fit, and Overfitting


In [ ]:
def fit_poly(x_train, y_train, degree):
    coeffs = np.polyfit(x_train, y_train, degree)
    return np.poly1d(coeffs)

degrees = [1, 3, 15]
x_grid = np.linspace(X.min(), X.max(), 500)

for degree in degrees:
    model = fit_poly(X_train, y_train, degree)

    plt.figure(figsize=(8,5))
    plt.scatter(X_train, y_train, label="Training data")
    plt.scatter(X_test, y_test, label="Test data")
    plt.plot(x_grid, model(x_grid), label=f"Polynomial degree {degree}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Model Complexity: Degree {degree}")
    plt.legend()
    plt.grid(alpha=.25)
    plt.show()


A low-degree model may be too rigid to capture the structure.

A very high-degree model may follow noise and local irregularities in the training sample.

The objective is not to fit the training data as closely as possible. The objective is to generalize.


## 7. Training and Test Error Across Complexity


In [ ]:
def mse(y, pred):
    return np.mean((y - pred)**2)

degree_grid = range(1, 19)
train_mse = []
test_mse = []

for degree in degree_grid:
    model = fit_poly(X_train, y_train, degree)
    train_mse.append(mse(y_train, model(X_train)))
    test_mse.append(mse(y_test, model(X_test)))

plt.figure(figsize=(8,5))
plt.plot(list(degree_grid), train_mse, marker="o", label="Training MSE")
plt.plot(list(degree_grid), test_mse, marker="o", label="Test MSE")
plt.xlabel("Polynomial degree")
plt.ylabel("Mean Squared Error")
plt.title("Training Error and Test Error vs Model Complexity")
plt.legend()
plt.grid(alpha=.25)
plt.show()

best_degree = list(degree_grid)[int(np.argmin(test_mse))]
print("Lowest observed test MSE at degree:", best_degree)


The training error generally decreases as flexibility increases.

The test error can decrease initially and then increase as the model begins fitting sample-specific noise.

That pattern is a classical signature of overfitting.


## 8. Underfitting

Underfitting occurs when a model is too restrictive to capture important structure in the data.

Typical evidence includes:

- high training error
- high validation/test error
- limited improvement even on the training data
- systematic residual structure


The approximation above is conceptual rather than a universal identity.


## 9. Overfitting

Overfitting occurs when a model captures training-specific patterns that do not persist in new data.

A common empirical pattern is:


Overfitting is not simply “a complex model.” Complexity becomes a problem when the fitted structure does not generalize.


## 10. Bias–Variance Decomposition

For squared-error prediction under a classical decomposition, the expected prediction error at x can be written as:


More explicitly:

- **Bias²** measures systematic error from the average learned model relative to the true function.
- **Variance** measures how much the fitted model changes across different training samples.
- **σ²** represents irreducible noise under the assumed data-generating model.

This decomposition depends on squared loss and the usual statistical assumptions.


## 11. Bias

Bias at an input x can be represented as:


A high-bias model is systematically too simple or incorrectly specified relative to the underlying relationship.


## 12. Variance

Model variance at x can be represented as:


A high-variance model changes substantially when the training sample changes.


## 13. Experiment — Visualizing Model Variance

We repeatedly draw new training samples from the same underlying process and fit models of different complexity.


In [ ]:
rng = np.random.default_rng(123)

def true_function(x):
    return 0.5*x**3 - x

x_grid = np.linspace(-3, 3, 250)

def repeated_predictions(degree, repetitions=100, n_train=30):
    predictions = []
    for _ in range(repetitions):
        x = np.sort(rng.uniform(-3, 3, n_train))
        y = true_function(x) + rng.normal(0, 0.8, n_train)
        model = np.poly1d(np.polyfit(x, y, degree))
        predictions.append(model(x_grid))
    return np.array(predictions)

pred_d1 = repeated_predictions(1)
pred_d3 = repeated_predictions(3)
pred_d12 = repeated_predictions(12)

for degree, preds in [(1,pred_d1),(3,pred_d3),(12,pred_d12)]:
    mean_pred = preds.mean(axis=0)
    lower = np.quantile(preds, .10, axis=0)
    upper = np.quantile(preds, .90, axis=0)

    plt.figure(figsize=(8,5))
    plt.plot(x_grid, true_function(x_grid), label="True function")
    plt.plot(x_grid, mean_pred, label="Mean fitted prediction")
    plt.fill_between(x_grid, lower, upper, alpha=.25, label="10th–90th percentile")
    plt.xlabel("x")
    plt.ylabel("Prediction")
    plt.title(f"Sampling Variability of Degree {degree} Model")
    plt.legend()
    plt.grid(alpha=.25)
    plt.show()


Higher flexibility can reduce systematic bias while increasing sensitivity to the particular training sample.

This is the essence of the bias–variance trade-off.


## 14. The Bias–Variance Trade-Off

Conceptually:


and:


These are tendencies, not inviolable laws for every algorithm and dataset.


## 15. Irreducible Error

Even a perfect model of the conditional mean cannot generally eliminate random noise in the outcome.

If:


with:


then σ² represents noise that cannot be removed simply by increasing model complexity.


## 16. Why One Test Split Is Not Enough

A model's measured test performance can depend on which observations happen to appear in the test set.

Therefore:


A single split is one estimate of future performance under particular sampling conditions.


## 17. Experiment — Instability Across Random Splits


In [ ]:
rng = np.random.default_rng(2026)

X_all = rng.uniform(-3, 3, 250)
y_all = true_function(X_all) + rng.normal(0, 0.8, len(X_all))

def random_split_mse(degree, repetitions=200, train_fraction=.7):
    values = []
    n = len(X_all)
    n_train = int(train_fraction*n)

    for _ in range(repetitions):
        idx = rng.permutation(n)
        train_idx = idx[:n_train]
        test_idx = idx[n_train:]

        model = np.poly1d(np.polyfit(X_all[train_idx], y_all[train_idx], degree))
        values.append(mse(y_all[test_idx], model(X_all[test_idx])))

    return np.array(values)

split_scores = random_split_mse(3)

print("Mean test MSE :", split_scores.mean())
print("Std test MSE  :", split_scores.std(ddof=1))
print("Min test MSE  :", split_scores.min())
print("Max test MSE  :", split_scores.max())

plt.figure(figsize=(8,5))
plt.hist(split_scores, bins=30)
plt.xlabel("Test MSE")
plt.ylabel("Frequency")
plt.title("Test Error Across Random Train/Test Splits")
plt.grid(alpha=.2)
plt.show()


The model did not fundamentally change. The measured evaluation changed because the sampled train/test partitions changed.


## 18. Cross-Validation

K-fold cross-validation partitions the available data into K folds.

Each fold is used once as validation data while the remaining folds are used for training.

If the validation loss on fold k is Lₖ, the cross-validation estimate is:


Cross-validation reduces dependence on a single split, but it does not remove every source of uncertainty or bias.


## 19. Experiment — K-Fold Cross-Validation Without Heavy Libraries


In [ ]:
def kfold_indices(n, k, rng):
    indices = rng.permutation(n)
    return np.array_split(indices, k)

def cv_mse(x, y, degree, k=5, seed=1):
    local_rng = np.random.default_rng(seed)
    folds = kfold_indices(len(x), k, local_rng)
    fold_losses = []

    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])

        model = np.poly1d(np.polyfit(x[train_idx], y[train_idx], degree))
        fold_losses.append(mse(y[val_idx], model(x[val_idx])))

    return np.array(fold_losses)

for degree in [1,2,3,5,10]:
    scores = cv_mse(X_all, y_all, degree, k=5, seed=42)
    print(f"degree={degree:>2} fold MSE={np.round(scores,3)} mean={scores.mean():.3f}")


## 20. Cross-Validation Is Still an Estimate

The cross-validation score depends on:

- the dataset
- the fold construction
- model-selection decisions
- preprocessing
- repeated experimentation
- the deployment distribution

It should therefore be treated as evidence with uncertainty, not as a mathematical guarantee.


## 21. Model Selection Bias

If many models are tested and the best one is chosen using the same validation data repeatedly, the selected score can become optimistic.

Conceptually:


The validation set can gradually become part of the model-development process through repeated human decisions.


## 22. Why a Final Test Set Exists

A final test set should ideally remain untouched until model selection and tuning are complete.

The intended separation is:


- Training data fits parameters.
- Validation data informs model and hyperparameter selection.
- Final test data estimates performance after decisions are fixed.


## 23. Data Leakage

Data leakage occurs when information unavailable at real prediction time improperly enters the training or model-selection process.

Leakage can produce apparently excellent validation results that do not generalize.


Examples include:

- future information entering past predictions
- target-derived variables appearing as inputs
- preprocessing fitted on the full dataset before splitting
- duplicate or near-duplicate records across train and test sets


## 24. Experiment — A Simple Leakage Demonstration

We construct a target and then deliberately create a feature that contains almost the answer itself.


In [ ]:
rng = np.random.default_rng(55)
n = 300

x_real = rng.normal(0,1,n)
y = 3*x_real + rng.normal(0,1,n)

# Leaky feature: target plus tiny noise
x_leak = y + rng.normal(0,.05,n)

idx = rng.permutation(n)
train = idx[:200]
test = idx[200:]

real_model = np.poly1d(np.polyfit(x_real[train], y[train], 1))
leak_model = np.poly1d(np.polyfit(x_leak[train], y[train], 1))

real_mse = mse(y[test], real_model(x_real[test]))
leak_mse = mse(y[test], leak_model(x_leak[test]))

print("MSE using legitimate feature:", round(real_mse,4))
print("MSE using leaked feature    :", round(leak_mse,4))


The leaky model can look extraordinary because the evaluation contains information that would not legitimately exist at prediction time.

Validation methodology must therefore examine the data pipeline, not just the final metric.


## 25. Learning Curves

A learning curve studies how training and validation error change as the amount of training data increases.

This helps distinguish data scarcity from model misspecification.


## 26. Experiment — Learning Curves


In [ ]:
rng = np.random.default_rng(77)
X_curve = rng.uniform(-3, 3, 500)
y_curve = true_function(X_curve) + rng.normal(0, 0.8, 500)

test_idx = np.arange(400,500)
X_fixed_test = X_curve[test_idx]
y_fixed_test = y_curve[test_idx]

train_sizes = [20,40,80,120,200,300,400]
train_losses=[]
test_losses=[]

for size in train_sizes:
    train_idx = np.arange(size)
    model = np.poly1d(np.polyfit(X_curve[train_idx], y_curve[train_idx], 3))
    train_losses.append(mse(y_curve[train_idx], model(X_curve[train_idx])))
    test_losses.append(mse(y_fixed_test, model(X_fixed_test)))

plt.figure(figsize=(8,5))
plt.plot(train_sizes, train_losses, marker="o", label="Training MSE")
plt.plot(train_sizes, test_losses, marker="o", label="Fixed-test MSE")
plt.xlabel("Training sample size")
plt.ylabel("MSE")
plt.title("Learning Curve")
plt.legend()
plt.grid(alpha=.25)
plt.show()


Learning curves can show whether more data is still improving generalization or whether model limitations dominate.


## 27. Generalization Requires Distributional Relevance

Even excellent cross-validation performance can fail to transfer if deployment data differ materially from the evaluation data.

Training distribution:


Deployment distribution:


If:


then performance measured under the training/evaluation distribution may not represent deployment performance.


## 28. Generalization Is Conditional

A scientifically defensible performance statement should specify the conditions under which generalization is expected.

A more careful statement is:

> The model achieved a measured performance level on data sampled under these conditions, using this evaluation procedure, with these observed uncertainties.

That is stronger than claiming:

> The model works.


## 29. Baseline vs Complex Model

Higher complexity is valuable only if it produces meaningful generalization improvement.

A simple baseline can protect against being impressed by complexity alone.


In [ ]:
rng=np.random.default_rng(88)
x=rng.uniform(-3,3,250)
y=true_function(x)+rng.normal(0,.8,250)
idx=rng.permutation(len(x))
train=idx[:170]
test=idx[170:]

baseline=np.full(len(test), y[train].mean())

results={}
for degree in [1,3,12]:
    model=np.poly1d(np.polyfit(x[train],y[train],degree))
    results[f"degree {degree}"]=mse(y[test],model(x[test]))

results["mean baseline"]=mse(y[test],baseline)

for name,value in results.items():
    print(f"{name:<14} test MSE = {value:.4f}")


## 30. Generalization Error Has Uncertainty

Even after choosing a model, the reported test error is a random quantity because the test sample is finite.

Therefore:


Resampling, repeated evaluation, confidence intervals, external validation, and monitoring can all help quantify different aspects of that uncertainty.


## 31. Scientific Validation Chain for Generalization


The goal is not merely to prove that a model can fit data.

The goal is to determine whether measured performance survives contact with new data.


## 32. OpenVals Perspective

Generalization is central to AI trust.

A validation system should preserve evidence about:

- train/validation/test separation
- model complexity
- training error
- validation error
- test error
- generalization gap
- repeated split variability
- cross-validation distributions
- leakage checks
- baseline comparisons
- learning curves
- subgroup performance
- external validation
- distribution shift
- performance drift after deployment

A single “accuracy” or “RMSE” value is not sufficient evidence of generalization.


## 33. Episode 05 Mathematical Summary


## 34. Central Conclusion


> **The purpose of Machine Learning is not to remember the past. It is to perform reliably on the future conditions for which it is intended.**

Training performance tells us how well a model fits observed data.

Generalization asks whether that performance survives on unseen data.

Validation therefore has to measure not only the score, but also the gap, sampling variability, model-selection process, leakage risk, and relevance of the evaluation distribution.

### Next Episode

**Episode 06 — Cross-Validation, Resampling, and How Much Evidence Is Enough?**

The next episode can go deeper into K-fold cross-validation, repeated cross-validation, bootstrap validation, confidence intervals, uncertainty of model comparisons, and statistically defensible model selection.
